In [1]:
# ============================================================
# PHASE 4 — ALS + Taste Signal (F1)
# Eval protocol: sampled 500 negatives, val split, rating>=4 positive
# ============================================================

!pip install implicit -q

import os
import json
import time
import random
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.preprocessing import normalize
from huggingface_hub import HfFileSystem, HfApi
from kaggle_secrets import UserSecretsClient
import implicit

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
HF_REPO  = "vngclinh/goodreads-preprocessed"
hf_fs    = HfFileSystem(token=HF_TOKEN)
api      = HfApi()

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

os.makedirs("/kaggle/working/phase4", exist_ok=True)

def hf_read(path, cols=None):
    with hf_fs.open(f"datasets/{HF_REPO}/{path}", "rb") as f:
        return pd.read_parquet(f, columns=cols)

print("Setup OK")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 18.8 MB/s eta 0:00:00
Setup OK


In [2]:
# ============================================================
# CELL 2 — Load artifacts
# ============================================================

user_taste = hf_read("phase3/user_taste_profile_f5_binary_positive.parquet")
book_vecs  = hf_read("lda/book_topic_vectors.parquet")

print("user_taste:", user_taste.shape)
print("book_vecs:", book_vecs.shape)

user_taste: (219385, 51)
book_vecs: (447813, 41)


In [3]:
# ============================================================
# CELL 3 — Load interactions với rating
# ============================================================

genres = [
    "children", "comics_-graphic", "fantasy_-paranormal", "fiction",
    "history_-historical-fiction_-biography", "mystery_-thriller_-crime",
    "non-fiction", "poetry", "romance", "young-adult"
]

parts = []
for g in genres:
    df = hf_read(f"data/{g}.parquet",
                 cols=["user_id", "book_id", "edge_weight", "split", "rating", "n_votes", "review_token_count"])

    parts.append(df)
    print(f"  Loaded: {g}")

interactions = pd.concat(parts, ignore_index=True)
del parts

interactions["user_id"] = interactions["user_id"].astype(str)
interactions["book_id"] = interactions["book_id"].astype(str)
interactions["rating"]  = pd.to_numeric(interactions["rating"], errors="coerce").fillna(0)
interactions["n_votes"] = pd.to_numeric(interactions["n_votes"], errors="coerce").fillna(0)
interactions["review_token_count"] = pd.to_numeric(interactions["review_token_count"], errors="coerce").fillna(0)

# F2
interactions["edge_weight"] = (interactions["rating"] >= 4).astype(float)

train_df = interactions[interactions["split"] == "train"].copy()
val_df   = interactions[interactions["split"] == "val"].copy()
test_df  = interactions[interactions["split"] == "test"].copy()

print(f"\nTrain: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")

  Loaded: children
  Loaded: comics_-graphic
  Loaded: fantasy_-paranormal
  Loaded: fiction
  Loaded: history_-historical-fiction_-biography
  Loaded: mystery_-thriller_-crime
  Loaded: non-fiction
  Loaded: poetry
  Loaded: romance
  Loaded: young-adult

Train: 9,483,169 | Val: 2,083,298 | Test: 1,317,468


In [4]:
# ============================================================
# CELL 4 — Build sparse matrix + index
# ============================================================

valid_users = set(user_taste["user_id"].astype(str).values)
train_df = train_df[train_df["user_id"].isin(valid_users)].copy()

all_users = sorted(train_df["user_id"].unique())
all_books = sorted(train_df["book_id"].unique())

user2idx = {u: i for i, u in enumerate(all_users)}
book2idx = {b: i for i, b in enumerate(all_books)}
idx2book = {i: b for b, i in book2idx.items()}

print(f"Users: {len(all_users):,} | Books: {len(all_books):,}")

rows = train_df["user_id"].map(user2idx).astype(int)
cols = train_df["book_id"].map(book2idx).astype(int)
vals = train_df["edge_weight"].astype(np.float32)

user_item = csr_matrix(
    (vals, (rows, cols)),
    shape=(len(all_users), len(all_books))
)
print(f"Matrix shape: {user_item.shape}, nnz: {user_item.nnz:,}")

Users: 219,385 | Books: 1,351,066
Matrix shape: (219385, 1351066), nnz: 9,483,169


In [5]:
# ============================================================
# CELL 5 — Train ALS
# ============================================================

model = implicit.als.AlternatingLeastSquares(
    factors=64,
    iterations=20,
    regularization=0.1,
    alpha=40,
    random_state=42,
    use_gpu=False
)

model.fit(user_item)
print("ALS training done")
print(f"User factors: {model.user_factors.shape}")
print(f"Item factors: {model.item_factors.shape}")

/usr/local/lib/python3.12/dist-packages/implicit/cpu/als.py:96: RuntimeWarning: OpenBLAS is configured to use 4 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


  0%|          | 0/20 [00:00<?, ?it/s]

ALS training done
User factors: (219385, 64)
Item factors: (1351066, 64)


In [6]:
# ============================================================
# CELL 6 — Prepare taste vectors
# ============================================================

topic_cols = [f"t{i}" for i in range(40)]

user_taste_indexed = user_taste.set_index("user_id")

book_vecs_indexed  = book_vecs.set_index("book_id")
book_topic_matrix  = normalize(
    book_vecs_indexed[topic_cols].values.astype(np.float32), norm="l2"
)
book_topic_bookids = book_vecs_indexed.index.tolist()
bookid2topicidx    = {b: i for i, b in enumerate(book_topic_bookids)}

print(f"book_topic_matrix: {book_topic_matrix.shape}")

book_topic_matrix: (447813, 40)


In [7]:
# ============================================================
# CELL 7 — Prepare val positives + train history
# ============================================================

# Positive = val rating >= 4
val_pos = val_df[val_df["rating"] >= 4].copy()
val_pos = val_pos[val_pos["user_id"].isin(valid_users)]

user_val_pos = val_pos.groupby("user_id")["book_id"].apply(list).to_dict()

# Train history để exclude khi sample negatives
user_train_books = train_df.groupby("user_id")["book_id"].apply(set).to_dict()

print(f"Users with val positives: {len(user_val_pos):,}")
print(f"Sample user pos count: {len(next(iter(user_val_pos.values())))}")

Users with val positives: 107,368
Sample user pos count: 4


In [8]:
# ============================================================
# CELL 8 — Metric functions + negative sampler
# ============================================================

def recall_at_k(recommended, relevant, k):
    return len(set(recommended[:k]) & set(relevant)) / min(len(relevant), k)

def ndcg_at_k(recommended, relevant, k):
    relevant_set = set(relevant)
    dcg  = sum(1 / np.log2(i + 2)
               for i, item in enumerate(recommended[:k])
               if item in relevant_set)
    idcg = sum(1 / np.log2(i + 2) for i in range(min(len(relevant), k)))
    return dcg / idcg if idcg > 0 else 0.0

all_book_ids = list(idx2book.values())

def sample_negatives(user_id, n_neg=500):
    seen      = user_train_books.get(str(user_id), set())
    positives = set(user_val_pos.get(str(user_id), []))
    forbidden = seen | positives
    negs, selected = [], set()
    trials = 0
    while len(negs) < n_neg and trials < 20000:
        b = random.choice(all_book_ids)
        trials += 1
        if b not in forbidden and b not in selected:
            selected.add(b)
            negs.append(b)
    return negs

print("Metric functions + sampler ready")

Metric functions + sampler ready


In [9]:
# ============================================================
# CELL 9 — Evaluate function (sampled protocol)
# ============================================================

def evaluate_sampled(user_val_pos, user2idx, book2idx, idx2book, model,
                     user_taste_indexed, book_topic_matrix, bookid2topicidx,
                     alpha=0.5, eval_k=(10, 20), n_neg=500):

    eval_users = [u for u in user_val_pos if u in user2idx]
    print(f"Evaluating {len(eval_users):,} users...")

    metrics = {f"Recall@{k}": [] for k in eval_k}
    metrics.update({f"NDCG@{k}": [] for k in eval_k})

    for idx, user_id in enumerate(eval_users):
        pos_books = [b for b in user_val_pos[user_id] if b in book2idx]
        if not pos_books:
            continue

        neg_books      = sample_negatives(user_id, n_neg=n_neg)
        candidate_books = list(dict.fromkeys(pos_books + neg_books))
        candidate_books = [b for b in candidate_books if b in book2idx]

        if not candidate_books:
            continue

        cand_idxs  = np.array([book2idx[b] for b in candidate_books])
        u_factor   = model.user_factors[user2idx[user_id]]
        als_scores = model.item_factors[cand_idxs] @ u_factor

        if alpha < 1.0 and user_id in user_taste_indexed.index:
            u_vec = user_taste_indexed.loc[user_id, topic_cols].values.astype(np.float32)
            u_vec = u_vec / (np.linalg.norm(u_vec) + 1e-9)

            taste_scores = np.full(len(candidate_books), np.nan)
            for j, b in enumerate(candidate_books):
                if b in bookid2topicidx:
                    taste_scores[j] = u_vec @ book_topic_matrix[bookid2topicidx[b]]

            mean_score   = np.nanmean(taste_scores)
            taste_scores = np.where(np.isnan(taste_scores),
                                    0.0 if np.isnan(mean_score) else mean_score,
                                    taste_scores)

            als_norm     = als_scores / (als_scores.max() + 1e-9)
            final_scores = alpha * als_norm + (1 - alpha) * taste_scores
        else:
            final_scores = als_scores

        ranked = [candidate_books[j] for j in np.argsort(-final_scores)]

        for k in eval_k:
            metrics[f"Recall@{k}"].append(recall_at_k(ranked, pos_books, k))
            metrics[f"NDCG@{k}"].append(ndcg_at_k(ranked, pos_books, k))

        if idx % 5000 == 0:
            print(f"  {idx:,} / {len(eval_users):,}")

    return {m: float(np.mean(v)) for m, v in metrics.items()}

print("evaluate_sampled ready")

evaluate_sampled ready


In [10]:
# ============================================================
# CELL 10 — Ablation trên val set
# ============================================================

variants = [
    {"alpha": 1.0, "name": "ALS_only"},
    {"alpha": 0.7, "name": "ALS_0.7_Taste_0.3"},
    {"alpha": 0.5, "name": "ALS_0.5_Taste_0.5"},
    {"alpha": 0.3, "name": "ALS_0.3_Taste_0.7"},
]

val_results = []
for v in variants:
    print(f"\n--- {v['name']} (alpha={v['alpha']}) ---")
    m = evaluate_sampled(
        user_val_pos, user2idx, book2idx, idx2book, model,
        user_taste_indexed, book_topic_matrix, bookid2topicidx,
        alpha=v["alpha"]
    )
    m["variant"] = v["name"]
    m["alpha"]   = v["alpha"]
    val_results.append(m)
    print(f"  Recall@10={m['Recall@10']:.4f} | NDCG@10={m['NDCG@10']:.4f} | "
          f"Recall@20={m['Recall@20']:.4f} | NDCG@20={m['NDCG@20']:.4f}")

val_df_results = pd.DataFrame(val_results)
best = val_df_results.loc[val_df_results["Recall@10"].idxmax()]
best_alpha = float(best["alpha"])
print(f"\n=== Best variant on val: {best['variant']} ===")
print(f"Recall@10={best['Recall@10']:.4f} | NDCG@10={best['NDCG@10']:.4f}")


--- ALS_only (alpha=1.0) ---
Evaluating 107,368 users...
  0 / 107,368
  5,000 / 107,368
  10,000 / 107,368
  15,000 / 107,368
  20,000 / 107,368
  25,000 / 107,368
  30,000 / 107,368
  40,000 / 107,368
  45,000 / 107,368
  50,000 / 107,368
  60,000 / 107,368
  65,000 / 107,368
  70,000 / 107,368
  75,000 / 107,368
  80,000 / 107,368
  85,000 / 107,368
  90,000 / 107,368
  95,000 / 107,368
  100,000 / 107,368
  105,000 / 107,368
  Recall@10=0.6161 | NDCG@10=0.5787 | Recall@20=0.6547 | NDCG@20=0.5868

--- ALS_0.7_Taste_0.3 (alpha=0.7) ---
Evaluating 107,368 users...
  0 / 107,368
  5,000 / 107,368
  10,000 / 107,368
  15,000 / 107,368
  20,000 / 107,368
  25,000 / 107,368
  30,000 / 107,368
  40,000 / 107,368
  45,000 / 107,368
  50,000 / 107,368
  60,000 / 107,368
  65,000 / 107,368
  70,000 / 107,368
  75,000 / 107,368
  80,000 / 107,368
  85,000 / 107,368
  90,000 / 107,368
  95,000 / 107,368
  100,000 / 107,368
  105,000 / 107,368
  Recall@10=0.6248 | NDCG@10=0.5826 | Recall@20=0.6

In [11]:
# ============================================================
# CELL 11 — Final evaluation trên test set (chạy 1 lần duy nhất)
# ============================================================

# Positive = test rating >= 4
test_pos = test_df[test_df["rating"] >= 4].copy()
test_pos = test_pos[test_pos["user_id"].isin(valid_users)]

user_test_pos = test_pos.groupby("user_id")["book_id"].apply(list).to_dict()
print(f"Users with test positives: {len(user_test_pos):,}")

# Override sample_negatives để dùng test positives làm forbidden
def sample_negatives_test(user_id, n_neg=500):
    seen      = user_train_books.get(str(user_id), set())
    positives = set(user_test_pos.get(str(user_id), []))
    forbidden = seen | positives
    negs, selected = [], set()
    trials = 0
    while len(negs) < n_neg and trials < 20000:
        b = random.choice(all_book_ids)
        trials += 1
        if b not in forbidden and b not in selected:
            selected.add(b)
            negs.append(b)
    return negs

def evaluate_sampled_test(user_test_pos, user2idx, book2idx, idx2book, model,
                          user_taste_indexed, book_topic_matrix, bookid2topicidx,
                          alpha=0.5, eval_k=(10, 20), n_neg=500):

    eval_users = [u for u in user_test_pos if u in user2idx]
    print(f"Evaluating {len(eval_users):,} users...")

    metrics = {f"Recall@{k}": [] for k in eval_k}
    metrics.update({f"NDCG@{k}": [] for k in eval_k})

    for idx, user_id in enumerate(eval_users):
        pos_books = [b for b in user_test_pos[user_id] if b in book2idx]
        if not pos_books:
            continue

        neg_books       = sample_negatives_test(user_id, n_neg=n_neg)
        candidate_books = list(dict.fromkeys(pos_books + neg_books))
        candidate_books = [b for b in candidate_books if b in book2idx]

        if not candidate_books:
            continue

        cand_idxs  = np.array([book2idx[b] for b in candidate_books])
        u_factor   = model.user_factors[user2idx[user_id]]
        als_scores = model.item_factors[cand_idxs] @ u_factor

        if alpha < 1.0 and user_id in user_taste_indexed.index:
            u_vec = user_taste_indexed.loc[user_id, topic_cols].values.astype(np.float32)
            u_vec = u_vec / (np.linalg.norm(u_vec) + 1e-9)

            taste_scores = np.full(len(candidate_books), np.nan)
            for j, b in enumerate(candidate_books):
                if b in bookid2topicidx:
                    taste_scores[j] = u_vec @ book_topic_matrix[bookid2topicidx[b]]

            mean_score   = np.nanmean(taste_scores)
            taste_scores = np.where(np.isnan(taste_scores),
                                    0.0 if np.isnan(mean_score) else mean_score,
                                    taste_scores)

            als_norm     = als_scores / (als_scores.max() + 1e-9)
            final_scores = alpha * als_norm + (1 - alpha) * taste_scores
        else:
            final_scores = als_scores

        ranked = [candidate_books[j] for j in np.argsort(-final_scores)]

        for k in eval_k:
            metrics[f"Recall@{k}"].append(recall_at_k(ranked, pos_books, k))
            metrics[f"NDCG@{k}"].append(ndcg_at_k(ranked, pos_books, k))

        if idx % 5000 == 0:
            print(f"  {idx:,} / {len(eval_users):,}")

    return {m: float(np.mean(v)) for m, v in metrics.items()}

print(f"Running final test eval with best_alpha={best_alpha}...")
test_metrics = evaluate_sampled_test(
    user_test_pos, user2idx, book2idx, idx2book, model,
    user_taste_indexed, book_topic_matrix, bookid2topicidx,
    alpha=best_alpha
)

print("\n=== FINAL TEST RESULTS ===")
for k, v in test_metrics.items():
    print(f"  {k}: {v:.4f}")

Users with test positives: 81,204
Running final test eval with best_alpha=0.7...
Evaluating 81,204 users...
  0 / 81,204
  5,000 / 81,204
  10,000 / 81,204
  20,000 / 81,204
  25,000 / 81,204
  30,000 / 81,204
  35,000 / 81,204
  40,000 / 81,204
  45,000 / 81,204
  50,000 / 81,204
  55,000 / 81,204
  60,000 / 81,204
  70,000 / 81,204
  75,000 / 81,204
  80,000 / 81,204

=== FINAL TEST RESULTS ===
  Recall@10: 0.5613
  Recall@20: 0.6278
  NDCG@10: 0.4995
  NDCG@20: 0.5189


In [12]:
# ============================================================
# CELL 12 — Ablation summary
# ============================================================

als_only_metrics = next(m for m in val_results if m["alpha"] == 1.0)

print("\n=== ABLATION (val set) ===")
print(f"{'Variant':<25} {'Recall@10':>10} {'NDCG@10':>10} {'Recall@20':>10} {'NDCG@20':>10}")
print("-" * 65)
for m in val_results:
    print(f"{m['variant']:<25} {m['Recall@10']:>10.4f} {m['NDCG@10']:>10.4f} "
          f"{m['Recall@20']:>10.4f} {m['NDCG@20']:>10.4f}")

print(f"\n=== FINAL TEST (best_alpha={best_alpha}) ===")
for k, v in test_metrics.items():
    print(f"  {k}: {v:.4f}")


=== ABLATION (val set) ===
Variant                    Recall@10    NDCG@10  Recall@20    NDCG@20
-----------------------------------------------------------------
ALS_only                      0.6161     0.5787     0.6547     0.5868
ALS_0.7_Taste_0.3             0.6248     0.5826     0.6748     0.5950
ALS_0.5_Taste_0.5             0.6099     0.5705     0.6571     0.5811
ALS_0.3_Taste_0.7             0.5668     0.5335     0.6144     0.5435

=== FINAL TEST (best_alpha=0.7) ===
  Recall@10: 0.5613
  Recall@20: 0.6278
  NDCG@10: 0.4995
  NDCG@20: 0.5189


In [13]:
# ============================================================
# CELL 13 — Save + Upload
# ============================================================

all_results = {
    "variant": "f5_binary_positive",
    "eval_protocol": "sampled_500_negatives_val_split",
    "positive_definition": "rating >= 4",
    "best_alpha": best_alpha,
    "val_ablation": val_results,
    "test_metrics": test_metrics,
}

metrics_path = "/kaggle/working/phase4/metrics_f5_als_sampled.json"
with open(metrics_path, "w") as f:
    json.dump(all_results, f, indent=2)

for attempt in range(3):
    try:
        api.upload_file(
            path_or_fileobj=metrics_path,
            path_in_repo = "phase4/metrics_f5_als_sampled.json",
            repo_id=HF_REPO, repo_type="dataset", token=HF_TOKEN
        )
        print("✓ metrics_f5_als_sampled.json uploaded")
        break
    except Exception as e:
        print(f"Attempt {attempt+1} failed: {e}")
        time.sleep(5)

print("\n=== PHASE 4 F5 COMPLETE ===")
print(f"Best alpha: {best_alpha}")
print("Test metrics:")
for k, v in test_metrics.items():
    print(f"  {k}: {v:.4f}")

✓ metrics_f5_als_sampled.json uploaded

=== PHASE 4 F5 COMPLETE ===
Best alpha: 0.7
Test metrics:
  Recall@10: 0.5613
  Recall@20: 0.6278
  NDCG@10: 0.4995
  NDCG@20: 0.5189
